In [ ]:
checkpoint_path = '/path/to/best_model'

In [ ]:


import pandas as pd
import numpy as np
import torch
from lightning import pytorch as pl
from pathlib import Path

from chemprop import data, featurizers, models



In [ ]:
mpnn = models.MPNN.load_from_checkpoint(checkpoint_path)
mpnn

In [ ]:
smiles_column = 'smiles'
df_test = pd.read_csv('path/to/train_or_test_data')
df_test

In [ ]:
smis = df_test[smiles_column]
smis

In [ ]:
test_data = [data.MoleculeDatapoint.from_smi(smi) for smi in smis]

In [ ]:
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()
test_dset = data.MoleculeDataset(test_data, featurizer=featurizer)
test_loader = data.build_dataloader(test_dset, shuffle=False)

In [ ]:
with torch.inference_mode():
    trainer = pl.Trainer(
        logger=None,
        enable_progress_bar=True,
        accelerator="cpu",
        devices=1
    )
    test_preds = trainer.predict(mpnn, test_loader)

In [ ]:
test_preds = np.concatenate(test_preds, axis=0)
df_test['pred'] = test_preds


In [ ]:
df_test['Ml_pred_exp'] = df_test['T_td'] + df_test['pred']

In [ ]:
df_test

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:
y_true = df_test["T_exp"]
y_pred = df_test["T_td"]

# R² score
r2 = r2_score(y_true, y_pred)

# RMSE (root mean squared error)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

# MAE (mean absolute error)
mae = mean_absolute_error(y_true, y_pred)

print(f"R²   : {r2:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

In [ ]:
y_true = df_test["T_exp"]
y_pred = df_test["Ml_pred_exp"]

# R² score
r2 = r2_score(y_true, y_pred)

# RMSE (root mean squared error)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

# MAE (mean absolute error)
mae = mean_absolute_error(y_true, y_pred)

print(f"R²   : {r2:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")

In [ ]:
df_test.to_csv('train_pred_td.csv', index=False)